# 02 - 信号诊断 / Signal Diagnostics

可视化 Z-Score 序列、开仓 / 平仓时刻、半衰期分布。

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from sqlalchemy import text

from core.data import IngestService
from strategy import generate_signals, screen_pairs
from strategy.zscore_signal import SignalConfig

sys.path.insert(0, str(Path.cwd().parent))

svc = IngestService()
prices = svc.load_ohlcv()
with svc.engine.begin() as conn:
    stocks = pd.read_sql(text("SELECT * FROM stocks"), conn)
pairs = screen_pairs(prices, stocks, pvalue_threshold=0.10, max_pairs=5)
pair = pairs[0]

In [ ]:
sig = generate_signals(prices, pair, cfg=SignalConfig())
fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
sig["z"].plot(ax=axes[0], color="steelblue")
axes[0].axhline(2.0, ls="--", c="r")
axes[0].axhline(-2.0, ls="--", c="r")
axes[0].axhline(0.5, ls=":", c="g")
axes[0].axhline(-0.5, ls=":", c="g")
axes[0].set_title(f"Z-Score - {pair.pair_id}")
sig["pos"].plot(ax=axes[1], drawstyle="steps-post", color="black")
axes[1].set_title("Position")
plt.tight_layout()